# UrbanVendor-Vision v2 · entrenamiento sin fugas y a escala nativa

<a href="https://colab.research.google.com/github/Kzamudioq/urbanvendor-vision/blob/main/notebooks/entrenar_v2_colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"></a>

**Runtime → Change runtime type → T4 GPU.** En Google Drive, la carpeta `UrbanVendor-Vision/etiquetas` debe tener las fotos `DJI_xxxx.JPG` y sus `DJI_xxxx.json` de Labelme, y `UrbanVendor-Vision/ortofo` la ortofoto.

Qué cambia frente a v1:

| | v1 | v2 |
|---|---|---|
| Escala de entrenamiento | foto reducida a 0,4× y luego a imgsz 1500 → 10,7 cm/píxel | teselas de 1024 px sin reducir → 3,6 cm/píxel |
| Validación | 2 fotos, una a 21 m y otra a 26 m de fotos de entrenamiento | 2 zonas del vuelo a más de 140 m: se entrena en una y se evalúa en la otra |
| Aumentos | 3 copias por foto en Roboflow | en cada época: rotación libre (vista nadir), volteos, color |
| Ortofoto | teselas de ~1024 px a imgsz 1500 → objetos 4,5× más grandes que en entrenamiento | ventanas de 1024 px a imgsz 1024 → misma escala |

In [1]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/UrbanVendor-Vision'
ETIQUETAS = f'{BASE}/etiquetas'
ORTO = f'{BASE}/ortofo/la_mariposa.tif'
SALIDA = f'{BASE}/v2'
!git clone -q https://github.com/Kzamudioq/urbanvendor-vision.git
%cd urbanvendor-vision
!pip install -q -e ".[modelo]"

Mounted at /content/drive
/content/urbanvendor-vision
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for urbanvendor (pyproject.toml) ... done


## 1. Escala de las fotos
El GSD sale de la altura y la cámara que guarda el dron en cada foto.

In [2]:
import glob
foto = sorted(glob.glob(f'{ETIQUETAS}/*.JPG'))[0]
!python -m urbanvendor escala --foto "$foto" --gsd-raster 0.0345453 --imgsz 1024

{
 "camara": "FC3170",
 "altura_m": 100.2,
 "gsd_foto_cm": 3.56,
 "gsd_modelo_cm": 3.56,
 "ventana_px": 1056
}


## 2. Dataset en teselas nativas, agrupado por zona

In [3]:
!python -m urbanvendor dataset --fotos "$ETIQUETAS" --salida /content/ds --lado 1024 --zona-m 140
!ls /content/ds

{
 "fotos": {
  "DJI_0674": {
   "zona": 0,
   "sombrillas": 77,
   "gsd_cm": 3.56
  },
  "DJI_0675": {
   "zona": 0,
   "sombrillas": 88,
   "gsd_cm": 3.56
  },
  "DJI_0679": {
   "zona": 0,
   "sombrillas": 48,
   "gsd_cm": 3.56
  },
  "DJI_0694": {
   "zona": 0,
   "sombrillas": 63,
   "gsd_cm": 3.56
  },
  "DJI_0729": {
   "zona": 1,
   "sombrillas": 208,
   "gsd_cm": 3.56
  },
  "DJI_0750": {
   "zona": 1,
   "sombrillas": 41,
   "gsd_cm": 3.56
  },
  "DJI_0755": {
   "zona": 1,
   "sombrillas": 17,
   "gsd_cm": 3.55
  },
  "DJI_0763": {
   "zona": 1,
   "sombrillas": 7,
   "gsd_cm": 3.56
  },
  "DJI_0765": {
   "zona": 1,
   "sombrillas": 3,
   "gsd_cm": 3.56
  },
  "DJI_0766": {
   "zona": 1,
   "sombrillas": 2,
   "gsd_cm": 3.56
  },
  "DJI_0767": {
   "zona": 1,
   "sombrillas": 1,
   "gsd_cm": 3.56
  },
  "DJI_0768": {
   "zona": 1,
   "sombrillas": 7,
   "gsd_cm": 3.56
  }
 },
 "zonas": 2
}
pliegue_0.yaml	pliegue_1.yaml	zona_0	zona_1


## 3. Validación cruzada por zona
Un modelo por pliegue: entrena con todas las zonas menos una y se evalúa en la que no vio. Esa es la cifra que se reporta.
`degrees=180` porque en vista nadir una sombrilla no tiene "arriba".

In [4]:
import glob, os, pandas as pd
from ultralytics import YOLO
HP = dict(imgsz=1024, epochs=150, patience=40, batch=8, degrees=180, flipud=0.5, fliplr=0.5,
          scale=0.25, mosaic=1.0, close_mosaic=15, cos_lr=True, seed=0, plots=True)
filas = []
for cfg in sorted(glob.glob('/content/ds/pliegue_*.yaml')):
    nombre = os.path.basename(cfg)[:-5]
    m = YOLO('yolo26s-seg.pt')
    m.train(data=cfg, project=f'{SALIDA}/cv', name=nombre, exist_ok=True, **HP)
    v = m.val(data=cfg, imgsz=1024, split='val', plots=False)
    filas.append({'pliegue': nombre, 'mAP50_mascara': v.seg.map50, 'mAP50_95_mascara': v.seg.map,
                  'precision': v.seg.mp, 'recall': v.seg.mr})
cv = pd.DataFrame(filas); cv.to_csv(f'{SALIDA}/validacion_cruzada.csv', index=False); cv

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/ds/pliegue_0.yaml, degrees=180, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze

,pliegue,mAP50_mascara,mAP50_95_mascara,precision,recall
0,pliegue_0,0.967076,0.783879,0.921873,0.940920
1,pliegue_1,0.975951,0.816044,0.927877,0.939341


## 4. Modelo final con todas las fotos
Las métricas que se publican son las de la validación cruzada; este modelo solo se usa para producir el mapa.

In [5]:
!mkdir -p /content/ds/todo
todo = '/content/ds/todo.yaml'
zonas = sorted(glob.glob('/content/ds/zona_*/images'))
open(todo, 'w').write('train:\n' + ''.join(f'  - {z}\n' for z in zonas) + f'val: {zonas[0]}\nnc: 1\nnames: [puesto_sombrilla]\n')
final = YOLO('yolo26s-seg.pt')
final.train(data=todo, project=SALIDA, name='final', exist_ok=True, **{**HP, 'patience': 0, 'epochs': 120})
PESOS = f'{SALIDA}/final/weights/last.pt'

Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/ds/todo.yaml, degrees=180, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=120, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=final, nbs=64, nms=None, opset=None, optimize=False, optimizer=au

## 5. Ortofoto completa y densidad
Ventanas de 1024 px a imgsz 1024: el modelo ve los objetos al mismo tamaño que en entrenamiento.

In [7]:
!python -m urbanvendor ortofoto --raster "$ORTO" --pesos "$PESOS" --imgsz 1024 --gsd-modelo 0.0356 --salida "$SALIDA/puestos_v2.geojson"
!python -m urbanvendor densidad --puestos "$SALIDA/puestos_v2.geojson" --lado 15 --salida "$SALIDA/densidad_v2.geojson"

GSD ortofoto 3.45 cm · modelo 3.56 cm · ventana 1055 px
  ventana 378/378
710 puestos -> /content/drive/MyDrive/UrbanVendor-Vision/v2/puestos_v2.geojson
{
 "puestos": 710,
 "diametro_mediano_m": 2.19,
 "hexagonos_con_puestos": 105,
 "hexagonos_totales": 532,
 "max_por_hexagono": 28,
 "hexagonos_con_la_mitad": 22
}
